# Assignment 5 - Decentralized Hyperparameter Learning of GPs

- **Topic:** Decentralized hyperparameter optimization for Gaussian Processes

- **Assessment:** The assignment will be graded and counted as part of the final assessment.


- **Deadline:** 4.8.2026
- **Submission: SUBMIT ONLY `assignment5_groupNumber.ipynb` TO BRIGHTSPACE**.


## Instructions
**Installation:** Use the `venvPSF` python kernel to ensure the python script runs as intended.

You may not use other packages for algorithm-related calculations.
You only need to complete (and submit) this file.

## AI Related Policy
We strongly discourage you to use AI tools for implementation assistance. It is your understanding of the problem that is tested in the final assessment.

## Information
Please fill in your group number, names and student numbers in the cell below.

In [ ]:
# YOUR ANSWER HERE

STUDENT_1_NAME = ""
STUDENT_1_STUDENT_NUMBER = ""

STUDENT_2_NAME = ""
STUDENT_2_STUDENT_NUMBER = ""

# remove or comment out this line
raise NotImplementedError()

## Objectives

In previous lectures and assignments, you learned that hyperparameters—such as the signal amplitude $\sigma_f$ and length scale $\ell$ in the squared exponential (SE) kernel—significantly impact Gaussian process (GP) regression. Therefore, it is natural to learn these hyperparameters directly from the data rather than tuning them heuristically.

This learning process is typically framed as an optimization problem where we maximize the likelihood of the data. However, as discussed in the lecture, this centralized approach becomes inconvenient for large datasets due to the inversion of a massive data covariance matrix, or for a dataset collected already in a decentralized fashion.

To address these issues, we will take a decentralized approach where we distribute the data across multiple agents. These agents will run smaller, local optimizations and communicate with one another so that their local updates ultimately converge to a centralized solution. 

Specifically, you will implement a **proximal Alternating Direction Method of Multipliers (pxADMM)** algorithm for the GP hyperparameter learning problem. If you recall, you already performed hyperparameter learning in Assignment 3, where you relied on a built-in second-order solver (BFGS) from SciPy. In this assignment, you will explicitly implement the pxADMM solver and its iterative updates yourself! 

*(Note: In Assignment 3, we optimized the parameters by maximizing the Marginal Log-Likelihood. In this assignment, we will frame the exact same objective as minimizing the Negative Log-Likelihood (NLL).)*

See the literature below for more mathematical details:

[1] A. Xie, F. Yin, Y. Xu, B. Ai, T. Chen and S. Cui, "Distributed Gaussian Processes Hyperparameter Optimization for Big Data Using Proximal ADMM," in IEEE Signal Processing Letters, vol. 26, no. 8, pp. 1197-1201, Aug. 2019.

[2] P. Zhai and R. T. Rajan, "Distributed Gaussian Process Hyperparameter Optimization for Multi-Agent Systems," ICASSP 2023 - 2023 IEEE International Conference on Acoustics, Speech and Signal Processing (ICASSP).



## 1. Graph Basics
Here we play with the fundamentals of graph theory. We first load our GP field in which the range of our "playground" is given. Then we generate some random locations for the agent and see if they form a connected graph based on some communication range.

## 1. Graph Basics

Before we dive into distributed optimization, we must establish the communication network between our agents. In graph theory, a network is represented as a graph $G=(V,E)$, consisting of a set of vertices $V$ (our agents) and a set of edges $E$ (the communication links). 

We first load our ground-truth GP field, which defines the boundaries of our 2D "playground". Next, we randomly scatter $N$ agents across this field. Two agents can communicate, and thus share an edge in the graph, if the Euclidean distance between them is less than or equal to a specified communication range.

In [ ]:
import numpy as np
import copy
import matplotlib.pyplot as plt
import scipy.io as sio
from tqdm import tqdm
from utils import param_array_to_dict, param_dict_to_array, visualize_graphs


# load artificial field data
gp_data = sio.loadmat('artificial_gp_field.mat')
gp_field = gp_data['gp_field'].squeeze()               # true field
field_resolution = gp_data['resolution'].squeeze()     # number of grid point in each dimension
field_range = gp_data['playground_range'].squeeze() 
print(f"Our Playground Range is: ", field_range.flatten())

### 1.1 Adjacency Matrix and Connectivity Check
Your task is to construct the **Adjacency Matrix**, $A$, which mathematically represents these connections. As discussed in the lecture, the entries of the adjacency matrix are defined as:
$$
A_{ij} = \begin{cases} 1, & \text{if distance}(i,j) \le \text{communication\_range} \text{ and } i \neq j \\ 0, & \text{otherwise} \end{cases}
$$
Once you construct the adjacency matrix, a helper function `is_graph_connected` is provided to verify if the entire graph is connected. You can tweak the communication range in the code below to test different network topologies and verify them using the provided graph visualization function.

**Question 1**: What does the adjacency matrix look like if there is a disconnected and isolated node?

**Answer**: 

**Task**: Implement the `calculate_adjacency_matrix` using the given `agent_locations` and `communication_range`.

In [ ]:
def calculate_adjacency_matrix(agent_locations, communication_range):
    """
    Calculate the adjacency matrix for a set of agents based on their communication range.
    
    Args:
        agent_locations (numpy.ndarray): Array of shape (GP_INPUT_DIM, NUM_AGENTS) 
                                         representing the coordinates of the agents.
        communication_range (float): The maximum Euclidean distance two agents can 
                                     be apart to establish a communication link.
                                     
    Returns:
        numpy.ndarray: A square integer array of shape (NUM_AGENTS, NUM_AGENTS) 
                       containing 1s for connected agents and 0s otherwise. 
                       The main diagonal should consist of zeros.
    """

    num_agents = agent_locations.shape[1]
    adjacency_matrix = np.zeros((num_agents, num_agents), dtype=int)

    
    ################# YOUR CODE HERE #################
    raise NotImplementedError()
    ##################################################

    return adjacency_matrix


from utils import is_graph_connected
NUM_AGENTS = 5
AGENT_COMM_RANGE = 5

# generate agent positions [GP_INPUT_DIM, NUM_AGENTS]
agent_locations = np.array([0.9*np.random.uniform(field_range[0,0], field_range[0,1], NUM_AGENTS),
                            0.9*np.random.uniform(field_range[1,0], field_range[1,1], NUM_AGENTS)])

adjacency_matrix = calculate_adjacency_matrix(agent_locations, AGENT_COMM_RANGE)
if not is_graph_connected(adjacency_matrix):
    print("Warning: The graph is not connected.")
else:
    print("The graph is connected.")
visualize_graphs(agent_locations, adjacency_matrix)

### 1.2 Architecture for Decentralized Optimization

As you have seen from the lecture, the standard Consensus ADMM algorithm requires all nodes to be commonly connected to a central data fusion node. This central node collects local information to compute the global consensus update and broadcasts it back. This is known as a decentralized (or star-topology) architecture.

**Question 2a**: If Agent 0 is designated as this central data fusion node, what specific pattern must exist in the 0-th row and column of the adjacency matrix?

**Question 2b**: Considering the physical deployment of these agents (e.g., a swarm of robots in a field), what are two practical limitations, communication constraints, or vulnerabilities of relying on this single central fusion node?

**Answer**: 


A variation of this architecture is the *fully distributed* version, where there is no central fusion node and consensus is reached purely through local neighbor-to-neighbor communications. For the scope of this assignment, we will implement the decentralized architecture. To simplify, we will assume an "invisible" central fusion node exists and is perfectly connected to all agents, so you do not have to worry about the connectivity constraints of the underlying graph for the rest of the implementation.

## 2. Proximal ADMM

Now we are going to build our pxADMM algorithm step by step. For debugging purposes, it is highly recommended to use a small-scale setup first (e.g., `NUM_AGENTS = 2` and `NUM_SAMPLES_PER_AGENT = 100`). **Warning:** Larger datasets will take significantly longer to process. Please ensure your "toy" version works before scaling up to test the algorithm's performance on massive data later in the assignment!

### 2.1 Simulation Setups

Here, we define the global settings for our optimization. 

**Note on Hyperparameters:** Our parameter vector is defined as $\boldsymbol{\theta} = [\sigma_f, l_1, l_2, \sigma_\epsilon]$. In the code below, these correspond to `signal_amplitude`, `length_scale` in 2 dimensions, and `noise_std`.

In [ ]:
GP_INPUT_DIM = 2
NUM_GP_HYPERPARAMS = 4
STOP_TOLERANCE = 1e-4     # stopping criterion for ADMM convergence
MAX_ITR = 15000           # maximum number of ADMM iterations

# true measurement noise standard deviation
noise_std = np.sqrt(0.1)      

# The true hyperparameters extracted from the artificial field data.
# We store these to evaluate our algorithm's convergence later.
gp_hyperparams_true = {
    'signal_amplitude': float(gp_data['sigma_f'].item()),   # sigma_f
    'length_scale': gp_data['l'].squeeze().astype(float),   # [l_1, l_2]
    'noise_std': float(noise_std)                           # sigma_epsilon
}

### 2.2 Generate Measurements

In this section, we will simulate the data collection process for our agents. 

First, we uniformly generate random sample locations within the given boundaries (`field_range`), we then generate clean measurements from the GP field and add Gaussian white noise to simulate real-world sensors with measurement model:
$$y_i = f(\mathbf{x}_i) + \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, \sigma_\epsilon^2)$$
where the standard deviation $\sigma_\epsilon$ is provided as `noise_std`, and $i$ is the sample index.

Finally, the code will uniformly distribute these noisy samples and their corresponding locations among the agents. *(Note: In a more realistic robotics deployment, agents would likely only collect data in their immediate physical proximity, rather than drawing a random subset from the entire field!)* The cell will output two figures: a contour plot of the true GP field with your generated samples overlaid, and a scatter plot showing how these samples are allocated to different agents in the network.

In [ ]:
NUM_AGENTS = 2
NUM_SAMPLES_PER_AGENT = 100

from utils import visualize_samples_on_field, visualize_sample_allocation, generate_data
# generate agent positions [GP_INPUT_DIM, NUM_AGENTS]
agent_locations = np.array([0.9*np.random.uniform(field_range[0,0], field_range[0,1], NUM_AGENTS),
                            0.9*np.random.uniform(field_range[1,0], field_range[1,1], NUM_AGENTS)])
sample_locations_reshaped, sample_locations_list, noisy_samples_reshaped, sample_locations, clean_samples, mesh_x1, mesh_x2 = generate_data(gp_field, 
                                                                                                                                            field_range, 
                                                                                                                                            field_resolution, 
                                                                                                                                            NUM_AGENTS, 
                                                                                                                                            NUM_SAMPLES_PER_AGENT, 
                                                                                                                                            noise_std)
# visualize the field and samples
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
contour = visualize_samples_on_field(axs[0], mesh_x1, mesh_x2, gp_field, sample_locations, clean_samples)
fig.colorbar(contour, ax=axs[0], label="Field Value")
visualize_sample_allocation(axs[1], agent_locations, sample_locations_list, adjacency_matrix)
plt.show()

### 2.3 Local Computations
We will first review the math behind the proximal ADMM (pxADMM) algorithm and then proceed to the code implementation.

As mentioned, GP hyperparameter optimization can be formulated as a likelihood maximization. Equivalently, we can frame this as minimizing the negative log-likelihood (NLL). In the context of ADMM, we distribute this by minimizing the sum of the local NLLs subject to a global consensus constraint:
$$
\begin{align}
  & \underset{\{\boldsymbol{\theta}_n\}_{n=1}^N,\mathbf{z}}{\textrm{min}}
  & & \sum_{n=1}^{N} l_n(\boldsymbol{\theta}_n) \\
  & \text{subject to}
  && \boldsymbol{\theta}_n = \mathbf{z},\quad n=1,...,N
\end{align}
$$
where $n=1,...,N$ is the agent index, and $\boldsymbol{\theta}_n$ are our local GP hyperparameters (representing the signal amplitude, length scales, and noise standard deviation). The local NLL has the form of:
$$
l_n(\boldsymbol{\theta}_n) = \frac{1}{2}\left(\mathbf{y}_n^{\mathsf{T}}\mathbf{K}_n^{-1}(\boldsymbol{\theta}_n)\mathbf{y}_n + \textrm{log}\left| \mathbf{K}_n(\boldsymbol{\theta}_n)  \right|\right)
$$
in which the full noisy data covariance matrix $\mathbf{K}_n$ consists of the Squared Exponential (SE) kernel values plus the noise variance, i.e.,
$$
k(\mathbf{x}_i,\mathbf{x}_j) = \sigma_f^2\textrm{exp}\left[ -\frac{1}{2}(\mathbf{x}_i-\mathbf{x}_j)^{\mathsf{T}}\boldsymbol{\Sigma}^{-1}(\mathbf{x}_i-\mathbf{x}_j)   \right] + \sigma_\epsilon^2\delta(\mathbf{x}_i-\mathbf{x}_j)
$$
where $\boldsymbol{\Sigma} = \textrm{diag}(l_1^2,l_2^2)$. Note that we use a different length-scale for each GP input dimension, which is a more general and flexible case.

To solve this distributively, we form the Augmented Lagrangian:
$$
    \mathcal{L}\left( \boldsymbol{\theta}_1,...,\boldsymbol{\theta}_N, \mathbf{z}, \boldsymbol{\lambda}_1,..., \boldsymbol{\lambda}_N\right) = \sum_{n=1}^{N}\left( l_n(\boldsymbol{\theta}_n)+ \boldsymbol{\lambda}_n^{\mathsf{T}}(\boldsymbol{\theta}_n-\mathbf{z}) + \frac{\rho}{2}\left\| \boldsymbol{\theta}_n-\mathbf{z}\right\|_2^2\right)
$$
where $\rho$ is a positive penalty constant and $\boldsymbol{\lambda}_1,..., \boldsymbol{\lambda}_N$ are the dual variables.

Following the standard ADMM framework, we minimize this Lagrangian by alternating the updates, namely, primal update, $z$-update, then the dual update: (the order can be flexible but we stick to the convention of the lecture)
$$
\begin{align}
\boldsymbol{\theta}_n^{t+1} & = \underset{\boldsymbol{\theta}_n}{\textrm{argmin}} \left(  l_n(\boldsymbol{\theta}_n)+ {\boldsymbol{\lambda}^t_n}^{\mathsf{T}}\boldsymbol{\theta}_n + \frac{\rho}{2}\left\| \boldsymbol{\theta}_n-\mathbf{z}^{t}\right\|_2^2\right)\\
\mathbf{z}^{t+1} &= \underset{\mathbf{z}}{\textrm{argmin}} \left( \sum_{n=1}^{N} \left( -{\boldsymbol{\lambda}^t_n}^{\mathsf{T}}\mathbf{z} + \frac{\rho}{2}\left\| \boldsymbol{\theta}^{t+1}_n-\mathbf{z}\right\|_2^2 \right)\right)\\
\boldsymbol{\lambda}_n^{t+1} &= \boldsymbol{\lambda}_n^{t} + \rho(\boldsymbol{\theta}_n^{t+1}-\mathbf{z}^{t+1})
\end{align}
$$

The primal update contains the NLL, and an analytical update is difficult to find directly. We linearize the NLL using a first-order Taylor expansion around $\mathbf{z}^t$:
$$
    l_n(\boldsymbol{\theta}_n) \approx l_n(\mathbf{z}^{t}) + \nabla^{\mathsf{T}} l_n(\mathbf{z}^{t})(\boldsymbol{\theta}_n-\mathbf{z}^{t})
$$
Substituting this into the augmented Lagrangian, the proximal primal update becomes:
$$
\boldsymbol{\theta}_n^{t+1}  = \underset{\boldsymbol{\theta}_n}{\textrm{argmin}} \left(  l_n(\mathbf{z}^{t}) + \nabla^{\mathsf{T}} l_n(\mathbf{z}^{t})(\boldsymbol{\theta}_n-\mathbf{z}^{t})+ {\boldsymbol{\lambda}^t_n}^{\mathsf{T}}\boldsymbol{\theta}_n + \frac{\rho+L}{2}\left\| \boldsymbol{\theta}_n-\mathbf{z}^{t}\right\|_2^2\right)
$$
where an additional quadratic penalty term $\frac{L}{2}\left\| \boldsymbol{\theta}_n-\mathbf{z}^{t}\right\|_2^2$ is added to increase algorithmic stability, and $L$ is a positive constant (often related to the Lipschitz constant of the gradient).


**Question 3**: What is the analytical solution to the $\mathbf{z}$-update? Give necessary steps.

**Answer**: 


**Question 4**: Give the analytical solution to the proximal primal update above with necessary steps. *(Hint: Many terms are with respect to $\mathbf{z}^{t}$ and act as constants that do not impact your optimal $\boldsymbol{\theta}_n$)*

**Answer**:

As you noticed, the term $\nabla l_n(\mathbf{z}^{t})$ arises which makes the computation of the gradients necessary, but you don't have to implement this your self. Now let's resume the coding.

<!-- 
From the NLL function, we directly give you the Jacobian
$$
\begin{equation}
\nabla l_m(\boldsymbol{\theta}) = \frac{1}{2}\textrm{tr}\left((\mathbf{K}_m^{-1}- \mathbf{K}_m^{-1} \mathbf{y}_m \mathbf{y}_m^{\mathsf{T}} \mathbf{K}_m^{-1})\frac{\partial\mathbf{K}_m(\boldsymbol{\theta})}{\partial\boldsymbol{\theta}}  \right)
\end{equation}
$$
from which you could further compute the gradients for each hyperparameter. As an excercise, we ask you to derive the gradients for $\sigma_f$ and $\noise_std$, and those for $l_1$ and $l_2$ will be given in the code.

**Question 5**: Derive $\frac{\partial\mathbf{K}_m(\boldsymbol{\theta})}{\partial\sigma_f}$ and $\frac{\partial\mathbf{K}_m(\boldsymbol{\theta})}{\partial\noise_std}$.

**Answer**: 
\begin{align}
\frac{\partial\mathbf{K}_m(\boldsymbol{\theta})}{\partial\sigma_f} &= \frac{2}{\sigma_f}\mathbf{K}_m(\boldsymbol{\theta})\\
\frac{\partial\mathbf{K}_m(\boldsymbol{\theta})}{\partial\noise_std} &= 2\left(\mathbf{K}^n_{m}(\boldsymbol{\theta})\right)^{\frac{1}{2}}
\end{align} -->











<!-- The local negative log-likelihood (NLL) is:
$$
l_m(\boldsymbol{\theta}) = \frac{1}{2}\left(\mathbf{y}_m^{\mathsf{T}}\mathbf{K}_m^{-1}(\boldsymbol{\theta})\mathbf{y}_m + \textrm{log}\left| \mathbf{K}_m(\boldsymbol{\theta})  \right|\right)
$$
based on which we could compute the gradient of the NLL with respect to $\boldsymbol{\theta}$ (refer to the matrix cookbook eq.40, 43)
$$
\begin{align}
\nabla l_m(\boldsymbol{\theta}) & = -\frac{1}{2}\mathbf{y}_m^{\mathsf{T}}\mathbf{K}_m^{-1}(\boldsymbol{\theta})\frac{\partial\mathbf{K}_m(\boldsymbol{\theta})}{\partial\boldsymbol{\theta}}\mathbf{K}_m^{-1}(\boldsymbol{\theta})\mathbf{y}_m + \frac{1}{2}\textrm{tr}\left( \mathbf{K}_m^{-1}(\boldsymbol{\theta})\frac{\partial\mathbf{K}_m(\boldsymbol{\theta})}{\partial\boldsymbol{\theta}}\right)\\
&= \frac{1}{2}\textrm{tr}\left((\mathbf{K}_m^{-1}- \mathbf{K}_m^{-1} \mathbf{y}_m \mathbf{y}_m^{\mathsf{T}} \mathbf{K}_m^{-1})\frac{\partial\mathbf{K}_m(\boldsymbol{\theta})}{\partial\boldsymbol{\theta}}  \right)
\end{align}
$$

Individually, the gradients for each hyperparameters are 
$$
\begin{align}
\frac{\partial\mathbf{K}_m(\boldsymbol{\theta})}{\partial\sigma_f} &= \frac{2}{\sigma_f}\mathbf{K}_m(\boldsymbol{\theta})\\
\frac{\partial\mathbf{K}_m(\boldsymbol{\theta})}{\partial l_1} &= \frac{\sigma_f^2}{l_1^3}\mathbf{K}^s_{m}(\boldsymbol{\theta})\odot\textrm{dist}(\mathbf{X}_{m,1})\\
\frac{\partial\mathbf{K}_m(\boldsymbol{\theta})}{\partial l_2} &= \frac{\sigma_f^2}{l_2^3}\mathbf{K}^s_{m}(\boldsymbol{\theta})\odot\textrm{dist}(\mathbf{X}_{m,2})\\
\frac{\partial\mathbf{K}_m(\boldsymbol{\theta})}{\partial\noise_std} &= 2\left(\mathbf{K}^n_{m}(\boldsymbol{\theta})\right)^{\frac{1}{2}}
\end{align}
$$
where $\odot$ is the Hadarmard product, $\textrm{dist}(\mathbf{X}_{m,1})$ gives the squared Euclidean distance matrix using only the 1st dimension of the sample locations. Note that the covariance matrix $\mathbf{K}_m(\boldsymbol{\theta})$ can be decomposed to the signal covariance and noise covariance $\mathbf{K}_m(\boldsymbol{\theta}) = \mathbf{K}_m^s(\boldsymbol{\theta})+\mathbf{K}^n_m(\boldsymbol{\theta})$ with the corresponding kernal
$$
k(\mathbf{x}_i,\mathbf{x}_j) = \sigma_f^2\textrm{exp}\left[ -\frac{1}{2}(\mathbf{x}_i-\mathbf{x}_j)^{\mathsf{T}}\boldsymbol{\Sigma}^{-1}(\mathbf{x}_i-\mathbf{x}_j)   \right] + \noise_std^2\delta(\mathbf{x}_i-\mathbf{x}_j)
$$
where $\boldsymbol{\Sigma} = \textrm{diag}(l_1^2,l_2^2)$.


The ADMM formulation is 
$$
\begin{align}
  & \underset{\{\boldsymbol{\theta}_m\}_{m=1}^M,\mathbf{z}}{\textrm{min}}
  & & \sum_{m=1}^{M} l_m(\boldsymbol{\theta}_m) \\
  & \text{subject to}
  && \boldsymbol{\theta}_m = \mathbf{z},\quad m=1,...,M\\
\end{align}
$$ -->
<!-- 
The augmented Lagragian is 
$$
    \mathcal{L}\left( \boldsymbol{\theta}_1,...,\boldsymbol{\theta}_M, \mathbf{z}, \boldsymbol{\beta}_1,..., \boldsymbol{\beta}_M\right) = \sum_{m=1}^{M}\left( l_m(\boldsymbol{\theta}_m)+ \boldsymbol{\beta}_m^{\mathsf{T}}(\boldsymbol{\theta}_m-\mathbf{z}) + \frac{\rho}{2}\left\| \boldsymbol{\theta}_m-\mathbf{z}\right\|_2^2\right)
$$
where $\rho$ is a positive constant and $\boldsymbol{\beta}_1,..., \boldsymbol{\beta}_M$ are the dual variables.

The ADMM update (z-update, primal update, and dual update) equations are :
$$
\begin{align}
\mathbf{z}^{t+1} &= \underset{\mathbf{z}}{\textrm{argmin}} \left( \sum_{m=1}^{M}-{\boldsymbol{\beta}^t_m}^{\mathsf{T}}\mathbf{z} + \frac{\rho}{2}\left\| \boldsymbol{\theta}_m-\mathbf{z}\right\|_2^2\right)\\
\boldsymbol{\theta}_m^{t+1} & = \underset{\boldsymbol{\theta}_m}{\textrm{argmin}} \left(  l_m(\boldsymbol{\theta}_m)+ {\boldsymbol{\beta}^t_m}^{\mathsf{T}}\boldsymbol{\theta}_m + \frac{\rho}{2}\left\| \boldsymbol{\theta}_m-\mathbf{z}\right\|_2^2\right)\\
\boldsymbol{\beta}_m^{t+1} &= \boldsymbol{\beta}_m^{t} + \rho(\boldsymbol{\theta}_m^{t+1}-\mathbf{z}^{t+1})

\end{align}
$$

The z-update is centrally computed can be solved analytically by
$$
\mathbf{z}^{t+1} = \frac{1}{M}\sum_{m=1}^{M}(\boldsymbol{\theta}_m^t+\frac{1}{\rho}\boldsymbol{\beta}_m^t)
$$

The primal update contains the NLL and a analytical update is difficult to find. As such, we linearize the NLL using the first order approximation
$$
    l_m(\boldsymbol{\theta}_m) \approx l_m(\mathbf{z}) + \nabla^{\mathsf{T}} l_m(\mathbf{z})(\boldsymbol{\theta}_m-\mathbf{z})
$$
such that the primal update becomes
$$
\boldsymbol{\theta}_m^{t+1}  = \underset{\boldsymbol{\theta}_m}{\textrm{argmin}} \left(  l_m(\mathbf{z}) + \nabla^{\mathsf{T}} l_m(\mathbf{z})(\boldsymbol{\theta}_m-\mathbf{z})+ {\boldsymbol{\beta}^t_m}^{\mathsf{T}}\boldsymbol{\theta}_m + \frac{\rho+L}{2}\left\| \boldsymbol{\theta}_m-\mathbf{z}\right\|_2^2\right)
$$
where $L$ is a constant. Solving this minimization gives the analytical primal update equation:
$$
    \theta_m^{t+1} = \mathbf{z}^{t+1} - \frac{1}{\rho+L}(\nabla l_m(\mathbf{z}^{t+1}) + \boldsymbol{\beta}^t_m)
$$ -->

### 2.4 Code Implementation: The ADMM Loop

Now we will translate the mathematical updates derived above into code. To make the architecture explicit, we have separated the algorithm into three distinct chronological steps that loop iteratively:

1. **Step 1: Local Primal Update ($\boldsymbol{\theta}_n^{t+1}$)**: Each agent evaluates the gradient of the Negative Log-Likelihood (NLL) at the *current* global consensus variable ($\mathbf{z}^t$). It then takes a proximal gradient step to update its local hyperparameters.
2. **Step 2: Global Consensus Update ($\mathbf{z}^{t+1}$)**: The central fusion node gathers the newly calculated local hyperparameters ($\boldsymbol{\theta}_n^{t+1}$) and the current dual variables ($\boldsymbol{\lambda}_n^t$) from all agents. It averages them to broadcast the *new* global consensus variable.
3. **Step 3: Local Dual Update ($\boldsymbol{\lambda}_n^{t+1}$)**: Each agent receives the new global consensus variable ($\mathbf{z}^{t+1}$) and updates its local dual variable (the "penalty" tracker) to enforce agreement in the next iteration.

**Task:**
Review the `Agent` class and the `pxADMM` main loop below, and complete the `pxADMM_primal_update()`, `pxADMM_dual_update()`, and the `pxADMM()` functions. Pay close attention to how `gp_hyperparams_local` maps to $\boldsymbol{\theta}_n$, `dual_var` maps to $\boldsymbol{\lambda}_n$, and `gp_hyperparams_global_dict` maps to $\mathbf{z}$.

In [ ]:
import copy
import numpy as np
from utils import compute_gp_gradient

class Agent:
    def __init__(self, agent_id, position, noisy_samples, sample_locations, sample_size, gp_input_dim=2):
        self.id = agent_id
        self.position = position
        self.input_dim = gp_input_dim

        # Dataset
        self.noisy_samples_local = noisy_samples
        self.sample_locations_local = sample_locations
        self.sample_size = sample_size
        assert(noisy_samples.shape[0] == sample_size)

        self.objective_local = 0
        self.gp_hyperparams_local = None
        self.pxADMM_params = None

    def pxADMM_primal_update(self, z_t_dict):
        """
        Step 1: Update local hyperparameters

        The gradient of the NLL is given with respect to each hyperparameter in the form of a dictionary,
        which contains the following keys:
        * 'partial_signal_amplitude' (float): The slope of the NLL w.r.t the signal amplitude.
        * 'partial_l' (np.ndarray): 1D array of shape (input_dim,) containing the 
            gradients w.r.t the length scales in each spatial dimension.
        * 'partial_noise_std' (float): The slope of the NLL w.r.t the measurement noise std.

        """
        # Call the utility function to get the NLL gradients evaluated at z^t
        gradients, self.objective_local = compute_gp_gradient(
            z_t_dict, 
            self.sample_locations_local, 
            self.noisy_samples_local, 
            self.sample_size, 
            self.input_dim
        )

        # Precompute the step size multiplier: 1 / (rho + L)
        step_size = 1.0 / (self.pxADMM_params['rho'] + self.pxADMM_params['L'])

        ################# YOUR CODE HERE #################
        raise NotImplementedError()
        new_signal_amplitude = ""
        new_l = ""
        new_noise_std = ""
        ##################################################

        # Store the updated local parameters
        self.gp_hyperparams_local['signal_amplitude'] = new_signal_amplitude
        self.gp_hyperparams_local['length_scale'] = new_l
        self.gp_hyperparams_local['noise_std'] = new_noise_std

    def pxADMM_dual_update(self, z_t_plus_1_dict):
        """
        Step 3: Update local dual variable (lambda_n^{t+1}) using the NEW global z^{t+1}.
        """
        # Convert dictionaries to arrays for vector math
        theta_array = param_dict_to_array(self.gp_hyperparams_local)
        z_array = param_dict_to_array(z_t_plus_1_dict)

        ################# YOUR CODE HERE #################
        raise NotImplementedError()
        self.pxADMM_params['dual_var'] += ""
        ##################################################

def pxADMM(agent_locations, 
           noisy_samples_reshaped, 
           sample_locations_reshaped, 
           num_samples_per_agent, 
           num_agents, max_itr, 
           stop_tolerance,
           init_gp_hyperparams = None,
           pxADMM_params = None):

    # Create logs for visualization
    gp_hyperparams_log = {
        'signal_amplitude': [],
        'length_scale': [],
        'noise_std': [],
    }
    objective_log = []      # Log the negative log-likelihood
    params_update_log = []  # Log the parameter diffs

    # Initialize agents
    agents = [Agent(i, agent_locations[:, i], noisy_samples_reshaped[:,i], sample_locations_reshaped[:,:,i], num_samples_per_agent) for i in range(num_agents)]
    for agent in agents: 
        if init_gp_hyperparams is not None and pxADMM_params is not None:
            agent.gp_hyperparams_local = copy.deepcopy(init_gp_hyperparams)
            agent.pxADMM_params = copy.deepcopy(pxADMM_params)
        else:
            agent.gp_hyperparams_local = {
                'signal_amplitude': 1.,
                'length_scale': np.array([2., 2.]),
                'noise_std': 1.
            }
            agent.pxADMM_params = {
                'dual_var': np.ones(NUM_GP_HYPERPARAMS),  # Dual variable (lambda_n)
                'rho': 400,                            # Consensus penalty parameter
                'L': 4000,                             # Lipschitz constant / stability term
            }

    print('pxADMM settings:', agents[0].pxADMM_params)

    itr = 0
    is_converged = False
    
    # Initialize global consensus variable z^0
    gp_hyperparams_global_dict = copy.deepcopy(agents[0].gp_hyperparams_local)
    old_gp_hyperparams_global_array = param_dict_to_array(gp_hyperparams_global_dict)

    with tqdm(total=max_itr) as pbar:
        while itr < max_itr and not is_converged:
            itr += 1

            # -----------------------------------------------------------------
            # STEP 1: Local Primal Update
            # -----------------------------------------------------------------
            for agent in agents:
                agent.pxADMM_primal_update(gp_hyperparams_global_dict)

            # -----------------------------------------------------------------
            # STEP 2: Global Data Fusion
            # -----------------------------------------------------------------
            new_z_array = np.zeros(NUM_GP_HYPERPARAMS)
            for agent in agents:
                theta_array = param_dict_to_array(agent.gp_hyperparams_local)
                
                ####################### YOUR CODE HERE ########################
                raise NotImplementedError()
                new_z_array += ""
                ###############################################################
            
            new_z_array /= num_agents
            gp_hyperparams_global_dict = param_array_to_dict(new_z_array)
            
            # -----------------------------------------------------------------
            # STEP 3: Local Dual Update
            # -----------------------------------------------------------------
            for agent in agents:
                agent.pxADMM_dual_update(gp_hyperparams_global_dict)

            # --- Logging and Convergence Checking ---
            objective_global = sum([agent.objective_local for agent in agents])
            objective_log.append(objective_global)

            gp_hyperparams_log['signal_amplitude'].append(new_z_array[0])
            gp_hyperparams_log['length_scale'].append(new_z_array[1:3])
            gp_hyperparams_log['noise_std'].append(new_z_array[3])

            param_diff = np.linalg.norm(new_z_array - old_gp_hyperparams_global_array)
            params_update_log.append(param_diff)
        
            if param_diff < stop_tolerance:
                is_converged = True
            
            old_gp_hyperparams_global_array = copy.deepcopy(new_z_array)

            pbar.update(1)
            pbar.set_postfix({"params updated by": param_diff})

    if is_converged:
        print(f'Converged after {itr} iterations')
    else:
        print(f'Not converged after {max_itr} iterations')

    print('Final hyperparameters:', param_array_to_dict(new_z_array))

    return gp_hyperparams_log, objective_log, params_update_log

### 2.5 Run pxADMM
After completing the local updates and the pxADMM main loop, let's run it and see the results!


In [ ]:
gp_hyperparams_log, objective_log, params_update_log = pxADMM(
    agent_locations, 
    noisy_samples_reshaped, 
    sample_locations_reshaped, 
    NUM_SAMPLES_PER_AGENT, 
    NUM_AGENTS, 
    MAX_ITR, 
    STOP_TOLERANCE
)

# visualize the convergence of the parameters
fig, axs = plt.subplots(3, 1, figsize=(8, 6), sharex=True)

# Updated from 'signal_var' to 'signal_amplitude'
axs[0].plot(gp_hyperparams_log['signal_amplitude'], label='signal_amplitude')
axs[0].axhline(y=gp_hyperparams_true['signal_amplitude'], color='r', linestyle='--', label='true signal_amplitude')
axs[0].set_ylabel('signal_amplitude')
axs[0].legend()

axs[1].plot([l[0] for l in gp_hyperparams_log['length_scale']], label='l1')
axs[1].plot([l[1] for l in gp_hyperparams_log['length_scale']], label='l2')
axs[1].axhline(y=gp_hyperparams_true['length_scale'][0], color='r', linestyle='--', label='true l1')
axs[1].axhline(y=gp_hyperparams_true['length_scale'][1], color='g', linestyle='--', label='true l2')
axs[1].set_ylabel('length_scale')
axs[1].legend()

axs[2].plot(gp_hyperparams_log['noise_std'], label='noise_std')
axs[2].axhline(y=gp_hyperparams_true['noise_std'], color='r', linestyle='--', label='true noise_std')
axs[2].set_ylabel('noise_std')
axs[2].legend()

plt.xlabel('Iteration')
plt.show()

# visualize the convergence of the objective and the parameters updates
fig, axs = plt.subplots(1, 2, figsize=(16, 4))

axs[0].plot(objective_log)
axs[0].set_xlabel('Iteration')
axs[0].set_ylabel('Negative Log Likelihood')

axs[1].plot(params_update_log)
axs[1].set_xlabel('Iteration')
axs[1].set_ylabel('Parameters Update')
axs[1].set_yscale('log')

plt.show()

Now let's reflect on the results.

**Question 5a**: Does the hyperparameters converge to the true values? How accurately do they converge, and what could be the cause if there is a gap?

**Answer**:

**Question 5b**: Do you see a monotonic convergence for the hyperparameters, the NLL, and the parameter updates? Comment on the reason.

**Answer**:

## 3 Learning from Magnetic Field Measurements
So far you have sucessfully implemented the pxADMM algorithm to learn the GP hyperparameters on a synthetic dataset. Now, let's try this algorithm on our classic magnetic field dateset that we used for Assignment 1 to 3. Like before, we pick a set of measurements and split it across several agents. Then we run our pxADMM algorithm to learn the hyperparameters and make some predictions.

**TASK**: Run the code below using the given initilization `init_gp_hyperparams`, and solver parameters `pxADMM_params` and observe the results. Then tweak these values and answer the following question.

**Question 6**: Is the estimated map close to the reference map? How sensitive is the algorithm to these parameters? Discuss you findings!

**Answer**:


In [ ]:
# Use the following settings to run once, then you can experiment with different settings to see how they affect convergence.
init_gp_hyperparams = {
            'signal_amplitude': 10.0,
            'length_scale': np.array([0.5, 0.5]),
            'noise_std': 0.5
        }

pxADMM_params = {
        'dual_var': np.ones(4),
        'rho': 1000,    
        'L': 1e5,     
    }

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy.linalg import inv
import helper 
import linAlg 

magnetometerMeasurements, magnetometerPositions, modelParameters = helper.initializeGaussianProcessAssignment(1)

# Calculate the true magnitude (norm) of the 3D magnetic field
magnetometerNorm = np.linalg.norm(magnetometerMeasurements, axis=0).reshape(-1, 1)


# Pre-process the Data for ADMM

# Calculate global mean of this file to zero-mean the GP
field_mean = np.mean(magnetometerNorm)
measurements_zero_mean = magnetometerNorm - field_mean

GP_INPUT_DIM = 2
positions_2d = magnetometerPositions[0:GP_INPUT_DIM, :]


# Distribute the Data Among the Agents
NUM_AGENTS = 10  
TOTAL_SAMPLES = measurements_zero_mean.shape[0]

NUM_SAMPLES_PER_AGENT = TOTAL_SAMPLES // NUM_AGENTS
valid_sample_count = NUM_AGENTS * NUM_SAMPLES_PER_AGENT

sliced_measurements = measurements_zero_mean[:valid_sample_count, :]
sliced_positions = positions_2d[:, :valid_sample_count]

# Reshape for the decentralized architecture: [Samples, Agents]
noisy_samples_reshaped = np.reshape(sliced_measurements, (NUM_SAMPLES_PER_AGENT, NUM_AGENTS), order='C')
sample_locations_reshaped = np.reshape(sliced_positions, (GP_INPUT_DIM, NUM_SAMPLES_PER_AGENT, NUM_AGENTS), order='C')

# Assign physical agent locations (the center of their respective assigned tracks)
agent_locations = np.mean(sample_locations_reshaped, axis=1)

print(f"Total dataset size: {valid_sample_count} (truncated to distribute evenly)")
print(f"Allocated {NUM_SAMPLES_PER_AGENT} continuous samples to each of the {NUM_AGENTS} agents.")


# Run pxADMM to Learn the Hyperparameters
MAX_ITR = 5000  
STOP_TOLERANCE = 5e-4

print("\nStarting Decentralized Optimization (pxADMM)...")
gp_hyperparams_log, objective_log, params_update_log = pxADMM(
    agent_locations, 
    noisy_samples_reshaped, 
    sample_locations_reshaped, 
    NUM_SAMPLES_PER_AGENT, 
    NUM_AGENTS, 
    MAX_ITR, 
    STOP_TOLERANCE,
    init_gp_hyperparams=init_gp_hyperparams,
    pxADMM_params=pxADMM_params
)

# Extract final parameters
final_signal_amplitude = gp_hyperparams_log['signal_amplitude'][-1]
final_l = gp_hyperparams_log['length_scale'][-1]
final_noise_std = gp_hyperparams_log['noise_std'][-1]


# Reconstruct the Magnetic Map using the Learned Parameters
print("\nReconstructing the magnetic map using learned parameters...")

# Flattening the data back for the global prediction
plot_pos = sample_locations_reshaped.reshape(GP_INPUT_DIM, -1, order='F')
plot_meas = noisy_samples_reshaped.reshape(-1, 1, order='F')
plot_true_norms = plot_meas + field_mean
n_plot_samples = valid_sample_count

posPred = linAlg.gridpointsHyperCube(60, 2, 3, np.array([[-0.4, 0.4], [-0.4, 0.4]]))
pred_positions_2d = posPred[0:2, :]

# Scale inputs
scaled_train_pos = np.diag(1.0 / final_l) @ plot_pos
scaled_pred_pos = np.diag(1.0 / final_l) @ pred_positions_2d

# Compute Covariances for GP Prediction
dist_train = cdist(scaled_train_pos.T, scaled_train_pos.T, 'sqeuclidean')
K = (final_signal_amplitude**2) * np.exp(-0.5 * dist_train) + (final_noise_std**2) * np.eye(n_plot_samples)

dist_cross = cdist(scaled_train_pos.T, scaled_pred_pos.T, 'sqeuclidean')
K_star = (final_signal_amplitude**2) * np.exp(-0.5 * dist_cross)

dist_pred = cdist(scaled_pred_pos.T, scaled_pred_pos.T, 'sqeuclidean')
K_star_star = (final_signal_amplitude**2) * np.exp(-0.5 * dist_pred)

# Invert K
K_inv = inv(K + 1e-8 * np.eye(n_plot_samples))

# Compute mean prediction (f) and covariance (cov)
f_zero_mean = K_star.T @ K_inv @ plot_meas
f = f_zero_mean + field_mean  
cov = K_star_star - K_star.T @ K_inv @ K_star


# Visualize the Final Result
helper.makeGaussianProcessMagneticFieldMapPlots(
    plot_pos, 
    posPred, 
    f, 
    cov, 
    plot_true_norms, 
    modelParameters
)
plt.suptitle(f"Magnetic Map: 1 Sensor File Split Across 10 Agents", y=1.02, fontsize=16)
plt.show()

**Feedback**:
Your feedback will be invaluable in improving it for the coming years. Please let us know in this markdown any comments, suggestions or errors you have encountered for this assignment.
